# Efficiency Comparison - CNN_LSTM vs MobileNetTransformer

Measures, per model:
- Total and trainable parameter counts.
- FLOPs for a single forward pass over one full clip (1, seq_len, 6, 224, 224), via `torch.profiler`. **Methodology note:** `torch.profiler`'s `with_flops` does not instrument `aten::lstm`, so CNN_LSTM's recurrent part is added back analytically (see `lstm_analytic_flops`) so the two models are compared on a like-for-like total.
- Inference latency (mean/std ms) and throughput (FPS) on CPU and CUDA (if available), for configurable batch sizes.

**Which checkpoints get compared** is controlled entirely by the `CNN_CKPT` / `MOBILENET_CKPT` cell below -- edit those paths to compare any pair of runs (baseline vs. ablation, or ablation vs. ablation), then **Run All**.

Each checkpoint's actual architecture (`pooling`, `transformer_layers`) is read back from the `config.json` saved next to it, so this notebook works correctly for baseline checkpoints and for any of the ablation checkpoints produced by `train/train_cnn_lstm.ipynb` / `train/train_mobilenet.ipynb`.

In [ ]:
import json
import os
import statistics
import sys
import time

import torch

BASE_DIR = os.path.abspath("..")
sys.path.append(BASE_DIR)

from models.cnn_lstm import CNN_LSTM
from models.mobile_net import MobileNetTransformer

SEQ_LEN = 20
IMG_SIZE = 224
IN_CHANNELS = 6  # RGB + motion channels, see utils/dataloader.py

print("BASE_DIR:", BASE_DIR)

## Configuration -- pick which two checkpoints to compare

In [ ]:
def latest_run_checkpoint(log_dir):
    runs = [d for d in os.listdir(log_dir) if d.startswith("run_") and os.path.isdir(os.path.join(log_dir, d))]
    if not runs:
        raise FileNotFoundError(f"No run_* folders found in {log_dir}")
    latest = sorted(runs)[-1]
    return os.path.join(log_dir, latest, "best_model.pth")

# Edit these two paths to compare any pair of checkpoints.
# picks the most recently created run for each model -- ALWAYS double-check
# the printed paths below actually point at the pair you intend to compare.
CNN_CKPT = latest_run_checkpoint(os.path.join(BASE_DIR, "outputs/logs/cnn_lstm"))
MOBILENET_CKPT = latest_run_checkpoint(os.path.join(BASE_DIR, "outputs/logs/mobile_net"))

# Latency measurement settings.
BATCH_SIZES = [1, 8]
WARMUP_ITERS = 5
TIMED_ITERS = 20

print("CNN_LSTM checkpoint   :", CNN_CKPT)
print("MobileNet checkpoint  :", MOBILENET_CKPT)

## Helper functions

In [ ]:
def get_num_classes(default=32):
    train_dir = os.path.join(BASE_DIR, "data/WLBisindo/split/train")
    if not os.path.isdir(train_dir):
        print(f"[warn] {train_dir} not found, falling back to default num_classes={default}")
        return default
    classes = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
    return len(classes) if classes else default


def load_run_config(ckpt_path):
    """config.json next to a checkpoint. Returns {} if missing/unreadable.
    NOTE: old baseline config.json files are only partially trustworthy (a
    stale-metadata bug once made one claim transformer_layers=2 while the
    checkpoint was actually trained with 4) -- resolve_mobilenet_arch()
    below only trusts a pooling value that is one of the three valid choices."""
    config_path = os.path.join(os.path.dirname(ckpt_path), "config.json")
    if not os.path.exists(config_path):
        return {}
    try:
        with open(config_path) as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def resolve_mobilenet_arch(ckpt_path):
    cfg = load_run_config(ckpt_path)
    pooling = cfg.get("pooling")
    if pooling not in ("last", "mean", "attention"):
        pooling = "last"
    num_layers = cfg.get("transformer_layers")
    if not isinstance(num_layers, int):
        num_layers = 4
    return pooling, num_layers


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def lstm_analytic_flops(input_size, hidden_size, num_layers, seq_len, batch_size):
    """Analytic FLOPs (1 multiply-add = 2 FLOPs) for a multi-layer
    unidirectional nn.LSTM forward pass, since torch.profiler doesn't
    instrument aten::lstm."""
    total = 0
    layer_input_size = input_size
    for _ in range(num_layers):
        per_step = 4 * (2 * layer_input_size * hidden_size + 2 * hidden_size * hidden_size)
        total += per_step * seq_len
        layer_input_size = hidden_size
    return total * batch_size


def measure_flops(model, sample_input, model_kind):
    model.eval()
    analytic_extra = 0
    notes = []

    if model_kind == "cnn_lstm":
        analytic_extra = lstm_analytic_flops(
            input_size=512, hidden_size=256, num_layers=2,
            seq_len=sample_input.shape[1], batch_size=sample_input.shape[0],
        )
        notes.append("Added analytic FLOPs for the 2-layer LSTM (input=512, hidden=256).")

    try:
        with torch.no_grad():
            with torch.profiler.profile(
                activities=[torch.profiler.ProfilerActivity.CPU],
                with_flops=True,
                record_shapes=True,
            ) as prof:
                model(sample_input)
        profiler_flops = sum(evt.flops for evt in prof.key_averages() if getattr(evt, "flops", None))
    except Exception as exc:
        profiler_flops = 0
        notes.append(f"torch.profiler FLOPs measurement failed ({exc!r}).")

    total = profiler_flops + analytic_extra
    return profiler_flops, analytic_extra, total, " ".join(notes)


def benchmark_latency(model, input_shape, device, warmup, iters):
    try:
        model = model.to(device)
        model.eval()
        x = torch.randn(*input_shape, device=device)

        with torch.no_grad():
            for _ in range(warmup):
                model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()

            times_ms = []
            for _ in range(iters):
                if device.type == "cuda":
                    start_evt = torch.cuda.Event(enable_timing=True)
                    end_evt = torch.cuda.Event(enable_timing=True)
                    start_evt.record()
                    model(x)
                    end_evt.record()
                    torch.cuda.synchronize()
                    times_ms.append(start_evt.elapsed_time(end_evt))
                else:
                    t0 = time.perf_counter()
                    model(x)
                    t1 = time.perf_counter()
                    times_ms.append((t1 - t0) * 1000.0)

        mean_ms = statistics.mean(times_ms)
        std_ms = statistics.pstdev(times_ms) if len(times_ms) > 1 else 0.0
        fps = 1000.0 / mean_ms if mean_ms > 0 else float("inf")
        result = {"mean_ms": mean_ms, "std_ms": std_ms, "fps": fps, "n_iters": iters}
    except RuntimeError as exc:
        result = {"error": str(exc)}
    finally:
        if device.type == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()

    return result

## Load models

In [ ]:
num_classes = get_num_classes()
cpu_device = torch.device("cpu")

cnn_model = CNN_LSTM(num_classes=num_classes)
cnn_model.load_state_dict(torch.load(CNN_CKPT, map_location=cpu_device, weights_only=True))

pooling, num_layers = resolve_mobilenet_arch(MOBILENET_CKPT)
print(f"MobileNetTransformer architecture (from config.json): pooling={pooling!r}, num_layers={num_layers}")
mobilenet_model = MobileNetTransformer(num_classes=num_classes, pooling=pooling, num_layers=num_layers)
mobilenet_model.load_state_dict(torch.load(MOBILENET_CKPT, map_location=cpu_device, weights_only=True))

devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))
print("Num classes:", num_classes)
print("Devices    :", [d.type for d in devices])

## Run the benchmark

In [ ]:
results = {
    "num_classes": num_classes,
    "cnn_ckpt": CNN_CKPT,
    "mobilenet_ckpt": MOBILENET_CKPT,
    "mobilenet_pooling": pooling,
    "mobilenet_num_layers": num_layers,
    "seq_len": SEQ_LEN,
    "img_size": IMG_SIZE,
    "methodology_note": (
        "FLOPs use the convention 1 multiply-add = 2 FLOPs. FLOPs for conv2d/linear/attention "
        "ops are measured via torch.profiler(with_flops=True); FLOPs for CNN_LSTM's nn.LSTM are "
        "added analytically (torch.profiler does not instrument aten::lstm). Latency is "
        "wall-clock for one forward pass (torch.cuda.Event on CUDA, time.perf_counter on CPU), "
        "mean +/- population std over TIMED_ITERS runs after WARMUP_ITERS warmup runs."
    ),
    "models": {},
}

for name, model, kind in [
    ("CNN_LSTM", cnn_model, "cnn_lstm"),
    ("MobileNetTransformer", mobilenet_model, "mobilenet_transformer"),
]:
    total_params, trainable_params = count_params(model)
    sample_input = torch.randn(1, SEQ_LEN, IN_CHANNELS, IMG_SIZE, IMG_SIZE)
    profiler_flops, analytic_extra, total_flops, note = measure_flops(model, sample_input, kind)

    entry = {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "flops_forward_single_clip": {
            "profiler_measured": profiler_flops,
            "analytic_extra": analytic_extra,
            "total": total_flops,
            "gflops": total_flops / 1e9,
            "note": note,
        },
        "latency": {},
    }

    for device in devices:
        for bs in BATCH_SIZES:
            key = f"{device.type}_bs{bs}"
            shape = (bs, SEQ_LEN, IN_CHANNELS, IMG_SIZE, IMG_SIZE)
            print(f"Benchmarking {name} on {device.type} (batch_size={bs}) ...")
            entry["latency"][key] = benchmark_latency(model, shape, device, WARMUP_ITERS, TIMED_ITERS)

    results["models"][name] = entry
    model.to(cpu_device)

print("Done.")

## Summary

In [ ]:
for name, entry in results["models"].items():
    print(f"\n{name}")
    print(f"  Params (total / trainable): {entry['total_params']:,} / {entry['trainable_params']:,}")
    print(f"  FLOPs (single clip, {SEQ_LEN} frames): {entry['flops_forward_single_clip']['gflops']:.3f} GFLOPs")
    for key, lat in entry["latency"].items():
        if "error" in lat:
            print(f"  Latency [{key}]: ERROR - {lat['error']}")
        else:
            print(f"  Latency [{key}]: {lat['mean_ms']:.2f} +/- {lat['std_ms']:.2f} ms  ({lat['fps']:.1f} FPS)")

## Save results

Saved with a filename derived from both checkpoints' run names, so comparing multiple pairs (baseline vs. ablation, ablation vs. ablation) doesn't overwrite previous results.

In [ ]:
cnn_run_name = os.path.basename(os.path.dirname(CNN_CKPT))
mobilenet_run_name = os.path.basename(os.path.dirname(MOBILENET_CKPT))

out_dir = os.path.join(BASE_DIR, "outputs/metrics")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"efficiency_comparison__{cnn_run_name}__vs__{mobilenet_run_name}.json")

with open(out_path, "w") as f:
    json.dump(results, f, indent=4)

print("Saved detailed results to:", out_path)